# 03 — Risk-limit impact: does a better volatility forecast catch more big moves?

**The desk's problem.** A derivatives desk sets next-week risk limits from the volatility it expects after an earnings release.
- **Under-forecasting** means excess risk and surprise limit breaches.
- **Over-forecasting** means idle capital.

Better volatility forecasts mean better limits. This notebook compares two forecasts at the job that matters most for limits: **picking out the events that will turn into big moves**.

- **B0, the desk's naive baseline:** past volatility only (`v_pre`, OLS).
- **B1b, our model:** past volatility plus the ranked size of the earnings surprise (`abs_sue_rank`), the signed surprise (winsorized), market cap, analyst coverage, sector and release timing (ridge, α = 1).

**Setup.** Both models are fit on dev (2006–16) and evaluated on validation (2017–19), on the same rows as the earlier baseline runs (84,940 dev / 25,659 validation). The target is `v_post`, the 5-day post-earnings log realized volatility. Holdout (2021–25) and 2020 are never loaded. No option or implied-volatility data is used: this compares forecast quality only.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))
import config as C
from baselines_b0_b1 import CAT_B1, NUM_B1, TARGETS, b0_model, load_locked, metrics
from baselines_b0_b1_fixed import ALPHA, SPECS, add_surprise_features, ridge_model

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
OUT_CSV = C.OUT / "results_risk_limit.csv"

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID, MISS = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#c9c8c2"
plt.rcParams.update({
    "figure.figsize": (9, 5), "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 14, "axes.titlepad": 26, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelsize": 11, "axes.labelcolor": INK2, "axes.edgecolor": MUTED,
    "text.color": INK, "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False, "text.parse_math": False,
})
MODEL_COLOR = {"B0": MUTED, "B1b": BLUE}

def subtitle(ax, s):
    ax.text(0, 1.012, s, transform=ax.transAxes, color=INK2, fontsize=10.5, va="bottom")

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", facecolor="white")
    plt.show()

## 1. Data and holdout lock

`load_locked()` (from `baselines_b0_b1.py`) reads only the dev and validation splits and raises if any other split, or any row after 2019, gets through. The assertions below repeat the check here.

In [ ]:
df = load_locked()
assert set(df["split"].unique()) <= {"dev", "validation"}, "HOLDOUT LOCK: unexpected split loaded"
assert df["session_date"].dt.year.max() <= 2019, "HOLDOUT LOCK: post-2019 row loaded"

df = df.dropna(subset=[*TARGETS, *NUM_B1, *CAT_B1]).copy()      # same rows as the baseline runs
for c in CAT_B1:
    df[c] = df[c].astype(str)
(dev, val), cuts = add_surprise_features(df[df["split"] == "dev"], df[df["split"] == "validation"])
assert (len(dev), len(val)) == (84_940, 25_659), "rows differ from the baseline runs"
print(f"dev {len(dev):,} | validation {len(val):,} | "
      f"validation sessions {val['session_date'].min().date()} to {val['session_date'].max().date()}")

## 2. Fit B0 and B1b on dev, predict validation

Same models as `baselines_b0_b1_fixed.py`: B0 is OLS on `v_pre`; B1b is ridge (α = 1, fixed, not tuned) with the dev-fitted surprise transforms.

In [ ]:
T = "v_post"
models = {"B0": b0_model(), "B1b": ridge_model(SPECS["B1b"])}
pred_dev, pred_val = {}, {}
for name, m in models.items():
    m.fit(dev, dev[T])
    pred_dev[name] = m.predict(dev)
    pred_val[name] = m.predict(val)
y_val = val[T].to_numpy()

acc = pd.DataFrame({n: metrics(y_val, pred_val[n], dev[T].mean()) for n in models}).T[["MAE", "RMSE", "R2_oos"]]
print("Statistical accuracy on validation, target v_post (R2_oos is measured against the dev-mean forecast)")
display(acc.round(4))

## 3. Define a "big move" on dev, then apply it to validation

A big move is an event whose realized `v_post` is in the **top quartile of dev**. The cut-off is computed once, on dev, and applied unchanged to validation (it is not recomputed on validation).

**Flag rule (as specified).** A model flags an event as *high risk* when its predicted `v_post` is above that same dev cut-off.

In [ ]:
LEVELS = {"top quartile": 0.75, "top decile": 0.90, "top tertile": 2 / 3}
thr = {k: dev[T].quantile(q) for k, q in LEVELS.items()}
for k, t in thr.items():
    print(f"{k:13s} dev cut-off v_post > {t:.3f}  =  daily vol > {np.exp(t):.2%}  "
          f"(share of validation events above it: {(y_val > t).mean():.1%})")

In [ ]:
def confusion(actual, flagged):
    tp = int(np.sum(actual & flagged)); fp = int(np.sum(~actual & flagged))
    fn = int(np.sum(actual & ~flagged)); tn = int(np.sum(~actual & ~flagged))
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn)
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn, "flag_rate": (tp + fp) / len(actual),
            "precision": prec, "recall": rec, "F1": 2 * prec * rec / (prec + rec),
            "false_alarm_rate": fp / (fp + tn)}

rows = []
for level, t in thr.items():
    actual = y_val > t
    for name in models:
        rows.append({"rule": "predicted > dev cut-off", "level": level, "model": name,
                     "cutoff_log": t, "cutoff_daily_vol": np.exp(t), "base_rate": actual.mean(),
                     **confusion(actual, pred_val[name] > t)})
dec = pd.DataFrame(rows)

main = dec[dec["level"] == "top quartile"].set_index("model")
print("Top-quartile big moves, validation (n = 25,659)")
display(main[["TP", "FP", "FN", "TN", "flag_rate", "precision", "recall", "F1", "false_alarm_rate"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"wspace": 0.45})
vmax = max(main[["TP", "FP", "FN", "TN"]].max())
for ax, name in zip(axes, ["B0", "B1b"]):
    r = main.loc[name]
    mat = np.array([[r["TP"], r["FN"]], [r["FP"], r["TN"]]])
    ax.imshow(mat, cmap="Blues", vmin=0, vmax=vmax * 1.15)
    lab = [["caught", "missed"], ["false alarm", "correctly calm"]]
    for i in range(2):
        for j in range(2):
            v = mat[i, j]
            ax.text(j, i, f"{v:,}\n{lab[i][j]}", ha="center", va="center", fontsize=12,
                    color="white" if v > vmax * 0.55 else INK, fontweight="bold" if (i, j) == (0, 0) else "normal")
    ax.set_xticks([0, 1], ["flagged high risk", "not flagged"])
    ax.set_yticks([0, 1], ["big move", "no big move"])
    ax.tick_params(length=0); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(f"{name}: {'past volatility only' if name == 'B0' else 'our model'}", fontsize=13, pad=10)
    ax.set_xlabel(f"recall {r['recall']:.1%} · precision {r['precision']:.1%} · false alarms {r['false_alarm_rate']:.1%}")
fig.suptitle("Who gets flagged: B0 vs B1b on validation big moves", x=0.06, ha="left", fontweight="bold", fontsize=15, y=1.04)
fig.text(0.06, 0.965, f"Validation 2017–19, n = {len(val):,}. Big move = realized v_post above the dev top-quartile cut-off "
         f"({np.exp(thr['top quartile']):.2%} daily vol); flagged = predicted v_post above the same cut-off.",
         color=INK2, fontsize=10, ha="left")
save(fig, "r1_confusion_matrices")

## 4. In the desk's language

Of the actual big-move events in validation, how many did each model catch? Expressed per 100 events and per quarter.

**Per-quarter assumption.** Validation covers 12 calendar quarters (2017 Q1 to 2019 Q4) and has 25,659 events in the primary sample, so about **2,138 earnings events per quarter**. This assumes the desk covers the whole primary-sample universe; scale linearly for a smaller book.

In [ ]:
N_Q = 12
ev_per_q = len(val) / N_Q
b0, b1 = main.loc["B0"], main.loc["B1b"]
n_big = b0["TP"] + b0["FN"]
per100 = lambda x: 100 * x / len(val)

desk = pd.DataFrame({
    "per 100 events": [per100(n_big), per100(b0["TP"]), per100(b1["TP"]), per100(b1["TP"] - b0["TP"]),
                       per100(b0["FP"]), per100(b1["FP"]), per100(b1["FP"] - b0["FP"])],
    "per quarter (approx.)": [n_big / N_Q, b0["TP"] / N_Q, b1["TP"] / N_Q, (b1["TP"] - b0["TP"]) / N_Q,
                              b0["FP"] / N_Q, b1["FP"] / N_Q, (b1["FP"] - b0["FP"]) / N_Q],
    "validation total": [n_big, b0["TP"], b1["TP"], b1["TP"] - b0["TP"], b0["FP"], b1["FP"], b1["FP"] - b0["FP"]],
}, index=["actual big moves", "caught by B0", "caught by B1b", "extra caught by B1b",
          "false alarms B0", "false alarms B1b", "extra false alarms B1b"])
print(f"Events per quarter assumed: {ev_per_q:,.0f} (= {len(val):,} validation events / {N_Q} quarters)")
display(desk.round(2))
print(f"B1b catches {per100(b1['TP'] - b0['TP']):.2f} more big moves per 100 events "
      f"(~{(b1['TP'] - b0['TP']) / N_Q:.0f} per quarter), at the cost of {per100(b1['FP'] - b0['FP']):.2f} more false alarms per 100.")
print(f"Relative: B1b catches {b1['TP'] / b0['TP'] - 1:.0%} more big moves than B0.")

In [ ]:
_big = y_val > thr["top quartile"]
eq_gain_q = per100(sum(np.sum(_big & (pred_val[n] > np.quantile(pred_val[n], 0.75))) * (1 if n == "B1b" else -1) for n in models))
fig, ax = plt.subplots(figsize=(9, 3.8))
y = [1, 0]
for yi, name in zip(y, ["B0", "B1b"]):
    r = main.loc[name]
    c, m = per100(r["TP"]), per100(r["FN"])
    ax.barh(yi, c, color=MODEL_COLOR[name], height=0.55, edgecolor="white", linewidth=2)
    ax.barh(yi, m, left=c, color=MISS, height=0.55, edgecolor="white", linewidth=2, alpha=0.6)
    ax.text(0, yi + 0.3, f"{c:.1f} caught", ha="left", va="bottom", color=INK, fontsize=11, fontweight="bold")
    ax.text(c + m / 2, yi, f"{m:.1f} missed", ha="center", va="center", color=INK, fontsize=11)
    ax.text(c + m + 0.4, yi, f"{per100(r['FP']):.1f} false alarms", va="center", color=INK2, fontsize=10)
ax.set_yticks(y, ["B0  past volatility", "B1b  our model"])
ax.set_xlim(0, per100(n_big) + 7)
ax.grid(axis="y", visible=False)
ax.set_xlabel(f"Big-move events per 100 earnings events (about {per100(n_big):.0f} in 100 are big moves)")
ax.set_ylim(-0.5, 1.75)
ax.set_title(f"Per 100 earnings events, B1b catches {per100(b1['TP'] - b0['TP']):.1f} more big moves than B0", pad=44)
subtitle(ax, f"Validation 2017–19. About {(b1['TP'] - b0['TP']) / N_Q:.0f} more per quarter at ~{ev_per_q:,.0f} events/quarter; "
             f"also {per100(b1['FP'] - b0['FP']):.1f} more false alarms per 100.\nWith equal flag counts the gain is {eq_gain_q:.2f} per 100 (see r4).")
save(fig, "r3_per_100_events")

## 5. Robustness: other big-move cut-offs

The same comparison at the top decile and top tertile of dev `v_post`, so the result is not an artifact of the quartile cut.

In [ ]:
order = ["top decile", "top quartile", "top tertile"]
rob = dec.pivot(index="level", columns="model", values=["recall", "precision", "flag_rate", "TP"]).reindex(order)
rob_tbl = pd.DataFrame({
    "cut-off (daily vol)": [f"{np.exp(thr[l]):.2%}" for l in order],
    "big moves in val": [int(dec[(dec.level == l)]["TP"].iloc[0] + dec[(dec.level == l)]["FN"].iloc[0]) for l in order],
    "recall B0": rob[("recall", "B0")].values, "recall B1b": rob[("recall", "B1b")].values,
    "extra caught per 100": per100(rob[("TP", "B1b")] - rob[("TP", "B0")]).values,
    "precision B0": rob[("precision", "B0")].values, "precision B1b": rob[("precision", "B1b")].values,
    "flag rate B0": rob[("flag_rate", "B0")].values, "flag rate B1b": rob[("flag_rate", "B1b")].values,
}, index=order)
display(rob_tbl.round(4))

In [ ]:
fig, ax = plt.subplots()
x = np.arange(len(order)); w = 0.36
for k, name in enumerate(["B0", "B1b"]):
    vals = rob[("recall", name)].values
    bars = ax.bar(x + (k - 0.5) * w, vals, w, color=MODEL_COLOR[name], edgecolor="white", linewidth=2,
                  label="B0  past volatility" if name == "B0" else "B1b  our model")
    for xi, v in zip(x + (k - 0.5) * w, vals):
        ax.text(xi, v + 0.005, f"{v:.1%}", ha="center", va="bottom", color=INK, fontsize=10.5)
ax.set_xticks(x, [f"{l}\n(> {np.exp(thr[l]):.1%} daily vol)" for l in order])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylim(0, rob[("recall", "B1b")].max() * 1.25)
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left")
ax.set_ylabel("Share of actual big moves flagged (recall)")
ax.set_xlabel("Definition of a big move (dev cut-off on realized v_post)")
ax.set_title("B1b catches more big moves than B0 at every cut-off")
subtitle(ax, "Validation 2017–19. Flagged = predicted v_post above the same dev cut-off.")
save(fig, "r2_catch_rate_thresholds")

## 6. Check: is B1b better at ranking, or does it just flag more?

Under the flag rule above, both models flag far fewer events than there are big moves. Regression forecasts are pulled towards the average, so few predictions reach a cut-off set on *realized* values. Recall is therefore low for both models, and B1b's higher recall comes partly from flagging more events: its predictions are more spread out.

A fairer test of forecast quality holds the number of flags exactly equal. Each model flags the same share of validation events as the big-move definition: its own top 25% (or 10%, 33%) of **predicted** `v_post` among the validation events. This uses only the predictions, never the realized outcomes, and it is how a desk would rank a set of upcoming events against each other. With identical flag counts, more catches can only come from better ranking.

(Cutting at each model's dev-prediction quantile instead would not give equal counts: validation is calmer before earnings than dev, so B0 would flag 17.2% and B1b 19.2% at the top-quartile level.)

In [ ]:
rows = []
for level, q in LEVELS.items():
    actual = y_val > thr[level]
    for name in models:
        pcut = np.quantile(pred_val[name], q)    # quantile of the model's own validation predictions (no labels)
        rows.append({"rule": "equal flag rate (top share of predictions)", "level": level, "model": name,
                     "cutoff_log": pcut, "cutoff_daily_vol": np.exp(pcut), "base_rate": actual.mean(),
                     **confusion(actual, pred_val[name] > pcut)})
eq = pd.DataFrame(rows)
eqp = eq.pivot(index="level", columns="model", values=["recall", "precision", "flag_rate", "TP"]).reindex(order)
eq_tbl = pd.DataFrame({
    "flag rate B0": eqp[("flag_rate", "B0")].values, "flag rate B1b": eqp[("flag_rate", "B1b")].values,
    "recall B0": eqp[("recall", "B0")].values, "recall B1b": eqp[("recall", "B1b")].values,
    "precision B0": eqp[("precision", "B0")].values, "precision B1b": eqp[("precision", "B1b")].values,
    "extra caught per 100": per100(eqp[("TP", "B1b")] - eqp[("TP", "B0")]).values,
}, index=order)
print("Equal flag rate: each model flags its own top share of validation predictions (identical flag counts)")
display(eq_tbl.round(4))

In [ ]:
fig, ax = plt.subplots()
for k, name in enumerate(["B0", "B1b"]):
    vals = eqp[("recall", name)].values
    ax.bar(x + (k - 0.5) * w, vals, w, color=MODEL_COLOR[name], edgecolor="white", linewidth=2,
           label="B0  past volatility" if name == "B0" else "B1b  our model")
    for xi, v in zip(x + (k - 0.5) * w, vals):
        ax.text(xi, v + 0.008, f"{v:.1%}", ha="center", va="bottom", color=INK, fontsize=10.5)
ax.set_xticks(x, [f"{l}\n(flags: top {1 - LEVELS[l]:.0%})" for l in order])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylim(0, eqp[("recall", "B1b")].max() * 1.2)
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left")
ax.set_ylabel("Share of actual big moves flagged (recall)")
ax.set_xlabel("Definition of a big move (dev cut-off on realized v_post)")
ax.set_title("With the same number of flags, B1b's edge is smaller but consistent")
subtitle(ax, "Validation 2017–19. Each model flags exactly the same number of events: its top share of predicted v_post.")
save(fig, "r4_catch_rate_equal_flags")

## 7. Save results

In [ ]:
acc_rows = acc.reset_index(names="model").assign(rule="accuracy (regression)", level="", target=T)
out = pd.concat([dec.assign(target=T), eq.assign(target=T), acc_rows], ignore_index=True)
out = out[["target", "rule", "level", "model", "cutoff_log", "cutoff_daily_vol", "base_rate", "TP", "FP", "FN", "TN",
           "flag_rate", "precision", "recall", "F1", "false_alarm_rate", "MAE", "RMSE", "R2_oos"]]
out.to_csv(OUT_CSV, index=False)
print(f"Saved {OUT_CSV} ({len(out)} rows)")
display(out.round(4))

## Deck-ready numbers

Every number below is computed in this notebook (cells above) and printed from the same variables, so it cannot drift from the figures.

In [ ]:
e = eq[eq.level == "top quartile"].set_index("model")
lines = f'''
**Sample and setup (applies to every number):** validation 2017–19, primary sample, n = {len(val):,} earnings events; models fit on dev 2006–16 (n = {len(dev):,}). Target: 5-day post-earnings log realized volatility (`v_post`). No holdout data used.

**Statistical accuracy (validation, v_post)**
- B0: MAE {acc.loc['B0','MAE']:.4f}, RMSE {acc.loc['B0','RMSE']:.4f}, out-of-sample R² {acc.loc['B0','R2_oos']:.4f}
- B1b: MAE {acc.loc['B1b','MAE']:.4f}, RMSE {acc.loc['B1b','RMSE']:.4f}, out-of-sample R² {acc.loc['B1b','R2_oos']:.4f}
- Improvement: R² +{acc.loc['B1b','R2_oos'] - acc.loc['B0','R2_oos']:.4f} ({acc.loc['B0','R2_oos']:.3f} → {acc.loc['B1b','R2_oos']:.3f}); RMSE −{acc.loc['B0','RMSE'] - acc.loc['B1b','RMSE']:.4f}; MAE −{acc.loc['B0','MAE'] - acc.loc['B1b','MAE']:.4f}
- Caveat: R² is measured against the dev-mean forecast.

**Big-move definition**
- Top quartile of dev realized `v_post`: v_post > {thr['top quartile']:.3f}, i.e. daily volatility > {np.exp(thr['top quartile']):.2%}. Fixed on dev, applied unchanged to validation.
- Share of validation events that are big moves: {main.loc['B0','base_rate']:.1%} ({int(n_big):,} events).

**Decision metrics: flag rule as specified (flag if predicted v_post > dev cut-off), top quartile**

| | B0 | B1b |
|---|---|---|
| Big moves caught (TP) | {int(b0['TP']):,} | {int(b1['TP']):,} |
| Big moves missed (FN) | {int(b0['FN']):,} | {int(b1['FN']):,} |
| False alarms (FP) | {int(b0['FP']):,} | {int(b1['FP']):,} |
| Correctly calm (TN) | {int(b0['TN']):,} | {int(b1['TN']):,} |
| Share of events flagged | {b0['flag_rate']:.1%} | {b1['flag_rate']:.1%} |
| Recall (catch rate) | {b0['recall']:.1%} | {b1['recall']:.1%} |
| Precision | {b0['precision']:.1%} | {b1['precision']:.1%} |
| F1 | {b0['F1']:.3f} | {b1['F1']:.3f} |
| False-alarm rate | {b0['false_alarm_rate']:.1%} | {b1['false_alarm_rate']:.1%} |

**Desk translation (top quartile, flag rule as specified)**
- Per 100 earnings events: about {per100(n_big):.1f} are big moves; B0 catches {per100(b0['TP']):.2f}, B1b catches {per100(b1['TP']):.2f}: **{per100(b1['TP'] - b0['TP']):.2f} more big moves caught per 100 events** ({b1['TP'] / b0['TP'] - 1:.0%} more than B0).
- Cost: {per100(b1['FP'] - b0['FP']):.2f} more false alarms per 100 events (B0 {per100(b0['FP']):.2f}, B1b {per100(b1['FP']):.2f}).
- Per quarter: **~{(b1['TP'] - b0['TP']) / N_Q:.0f} more big moves caught** and ~{(b1['FP'] - b0['FP']) / N_Q:.0f} more false alarms. Assumption: {ev_per_q:,.0f} events per quarter = {len(val):,} validation events / 12 quarters (2017 Q1–2019 Q4), i.e. a desk covering the whole primary-sample universe.

**Robustness across cut-offs (flag rule as specified)**

| Big move = | cut-off (daily vol) | recall B0 | recall B1b | extra caught per 100 |
|---|---|---|---|---|
''' + "\n".join(f"| {l} | {np.exp(thr[l]):.2%} | {rob.loc[l, ('recall','B0')]:.1%} | {rob.loc[l, ('recall','B1b')]:.1%} | {per100(rob.loc[l, ('TP','B1b')] - rob.loc[l, ('TP','B0')]):.2f} |" for l in order) + f'''

**Equal-flag-rate check (each model flags exactly the same number of events: its top share of predicted v_post among validation events; no outcome labels used)**

| Big move = | flags | recall B0 | recall B1b | precision B0 | precision B1b | extra caught per 100 |
|---|---|---|---|---|---|---|
''' + "\n".join(f"| {l} | top {1 - LEVELS[l]:.0%} | {eqp.loc[l, ('recall','B0')]:.1%} | {eqp.loc[l, ('recall','B1b')]:.1%} | {eqp.loc[l, ('precision','B0')]:.1%} | {eqp.loc[l, ('precision','B1b')]:.1%} | {per100(eqp.loc[l, ('TP','B1b')] - eqp.loc[l, ('TP','B0')]):.2f} |" for l in order) + f'''

**Caveats to state on the slides**
1. **Low absolute recall under the specified rule.** Both models flag far fewer events ({b0['flag_rate']:.1%} and {b1['flag_rate']:.1%}) than the {main.loc['B0','base_rate']:.1%} that are big moves, because regression forecasts are pulled towards the average. The rule catches the clearest cases only; a desk would set the flag cut-off on predictions, not on realized values.
2. **Most of B1b's gain under the specified rule comes from flagging more events.** Precision is about the same ({b0['precision']:.1%} vs {b1['precision']:.1%}) while B1b flags {b1['flag_rate']:.1%} of events against {b0['flag_rate']:.1%}. The equal-flag-rate check isolates ranking quality: at the top-quartile definition, with both models flagging exactly 25% of events, B1b catches **{per100(eqp.loc['top quartile', ('TP','B1b')] - eqp.loc['top quartile', ('TP','B0')]):.2f} more big moves per 100 events** (~{(eqp.loc['top quartile', ('TP','B1b')] - eqp.loc['top quartile', ('TP','B0')]) / N_Q:.0f} per quarter; recall {eqp.loc['top quartile', ('recall','B0')]:.1%} → {eqp.loc['top quartile', ('recall','B1b')]:.1%}). This is the defensible headline for "better forecasts"; the {per100(b1['TP'] - b0['TP']):.2f} per 100 figure mixes ranking and flag volume.
3. **Validation, not holdout.** B1b's design choices (ranked surprise) were made after looking at validation results, so these numbers are slightly optimistic. The holdout (2021–25) is the clean test and has not been run. The README notes the holdout period is more volatile than dev.
4. **Forecast quality only.** No implied volatility, option prices, P&L or capital costs are involved; "caught" and "false alarm" are forecast outcomes, not dollar figures.
5. **Per-quarter figures** assume the full primary-sample universe (~{ev_per_q:,.0f} events per quarter) and scale linearly with book size.
'''
display(Markdown(lines))